# 01 · Scoring rules

How a probability forecast is graded once the outcome is known. Everything here runs offline on simulated data.

In [ ]:
import sys

sys.path.insert(0, "../src")
import matplotlib.pyplot as plt
import numpy as np

from forecaster.scoring import brier_score, log_score

## Brier score

`(probability - outcome)^2`, averaged. 0 is perfect, 0.25 is what always saying 50% gets on a 50/50 question.

In [ ]:
print("always 50%:", brier_score([0.5], [1.0]), brier_score([0.5], [0.0]))
print("confident and right:", brier_score([0.95], [1.0]))
print("confident and wrong:", brier_score([0.95], [0.0]))

## Log score

`-log(probability you gave to what happened)`. Confident mistakes cost a lot more under log score than under Brier.

In [ ]:
print("Brier, confident wrong:", brier_score([0.99], [0.0]))
print("Log score, confident wrong:", log_score([0.99], [0.0]))

## Proper scoring rules reward honesty

If the true probability of YES is `true_p`, what report minimizes your *expected* score? Proper scoring rules are built so the answer is always `true_p` itself — there's no way to game them by hedging.

In [ ]:
true_p = 0.7
reports = np.linspace(0.01, 0.99, 99)
expected_brier = true_p * (reports - 1) ** 2 + (1 - true_p) * reports**2
plt.plot(reports, expected_brier)
plt.axvline(true_p, color="red", linestyle="--", label=f"true_p={true_p}")
best_report = reports[np.argmin(expected_brier)]
plt.axvline(best_report, color="green", linestyle=":", label=f"best report={best_report:.2f}")
plt.xlabel("reported probability")
plt.ylabel("expected Brier score")
plt.legend()
plt.title("Expected Brier score is minimized by reporting the truth")

**Try it:** change `true_p` to `0.9` above and re-run — the best report moves to 0.9 right along with it.

## A score means nothing without a baseline

A low Brier score on an easy question set (mostly 95/5 questions) doesn't mean the forecaster is skilled — a coin that always says 'the easy side' scores well too. Scores are only meaningful compared against a baseline on the *same* question set: always-50%, the base rate, or the market price.